In [1]:

import numpy as np
import pandas as pd
import re
df = pd.read_csv("csv/deliveries.csv")
matches = pd.read_csv("csv/matches.csv")

In [2]:
import pandas as pd

rename_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiant": "Rising Pune Supergiants"
}

# rename first
df['batting_team'] = df['batting_team'].replace(rename_map)

# Calculate runs per team per over
runs_team_over = df.groupby(['batting_team', 'over'])['total_runs'].sum()

# Normalize to 20 time resources
ow = (runs_team_over / runs_team_over.groupby(level=0).transform('sum')) * 20

# Convert to dataframe
ow_df = ow.reset_index()
ow_df.columns = ['batting_team', 'over', 'weight']

# Save
ow_df.to_csv('csv/over_weights.csv', index=False)

print("over_weights.csv saved successfully")

# check totals
print(ow_df.groupby('batting_team')['weight'].sum())

over_weights.csv saved successfully
batting_team
Chennai Super Kings            20.0
Deccan Chargers                20.0
Delhi Capitals                 20.0
Gujarat Lions                  20.0
Gujarat Titans                 20.0
Kochi Tuskers Kerala           20.0
Kolkata Knight Riders          20.0
Lucknow Super Giants           20.0
Mumbai Indians                 20.0
Pune Warriors                  20.0
Punjab Kings                   20.0
Rajasthan Royals               20.0
Rising Pune Supergiants        20.0
Royal Challengers Bengaluru    20.0
Sunrisers Hyderabad            20.0
Name: weight, dtype: float64


In [19]:

# remove numbering like (2)
for col in ['batter', 'bowler']:
    df[col] = df[col].str.replace(r'\s*\(\d+\)', '', regex=True)
    df[col] = df[col].str.strip()

# manual corrections
name_map = {
    'R Sharma': 'RG Sharma',
    'R Bishnoi': 'Ravi Bishnoi'
}

df[['batter','bowler']] = df[['batter','bowler']].replace(name_map)
innings = df.groupby(['match_id','batter']).agg(
    runs=('batsman_runs','sum'),
    out=('player_dismissed', lambda x: x.notna().sum())
).reset_index()


print(innings)

       match_id         batter  runs  out
0        335982      AA Noffke     9    1
1        335982        B Akhil     0    1
2        335982    BB McCullum   158    0
3        335982       CL White     6    1
4        335982      DJ Hussey    12    1
...         ...            ...   ...  ...
16503   1426312      SP Narine     6    1
16504   1426312        SS Iyer     6    0
16505   1426312  Shahbaz Ahmed     8    1
16506   1426312        TM Head     0    1
16507   1426312        VR Iyer    52    0

[16508 rows x 4 columns]


In [ ]:
innings = innings.sort_values(['batter','match_id'])
# Group by batter BEFORE shifting to prevent data leaking between different players
innings['cum_runs'] = innings.groupby('batter')['runs'].cumsum().groupby(innings['batter']).shift()
innings['cum_outs'] = innings.groupby('batter')['out'].cumsum().groupby(innings['batter']).shift()

# Replace 0 with NaN to avoid 'inf' values for players who haven't been out yet
innings['pavg'] = innings['cum_runs'] / innings['cum_outs'].replace(0, np.nan)


In [ ]:
#  Ensure the rolling window stays within the specific batter's history
innings['lfm'] = (
    innings.groupby('batter')['runs']
    .shift()
    .groupby(innings['batter']) # Group again to apply rolling per player
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
# 1 Merge venue
innings = innings.merge(
    matches[['id','venue']],
    left_on='match_id',
    right_on='id',
    how='left'
)
innings = innings.drop(columns=['id'], errors='ignore')

# 2 Clean venue names 
innings['venue'] = innings['venue'].str.split(',').str[0].str.strip()
innings['venue'] = innings['venue'].replace({
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    'Feroz Shah Kotla Ground': 'Arun Jaitley Stadium',
    'Dubai International Cricket Stadium': 'UAE',
    'Sharjah Cricket Stadium': 'UAE',
    'Sheikh Zayed Stadium': 'UAE',
    'Punjab Cricket Association Stadium': 'Punjab Cricket Association IS Bindra Stadium',
    'Punjab Cricket Association Bindra Stadium': 'Punjab Cricket Association IS Bindra Stadium'
})

# 3 Reduce rare stadiums 
venue_counts = innings['venue'].value_counts()
top_venues = venue_counts.nlargest(13).index
innings['venue'] = innings['venue'].where(
    innings['venue'].isin(top_venues),
    'Other'
)

# 4 Sort before expanding averages 
# Adding 'match_id' or a date column ensures the 'expanding' window respects time.
innings = innings.sort_values(['batter', 'match_id'])

# 5 Compute stadium average
# Using .transform() instead of .reset_index() ensures the values 
# stay mapped to the correct rows. reset_index(drop=True) often causes 
# data to be assigned to the wrong batter if the index order changed.
innings['savg'] = (
    innings.groupby(['batter', 'venue'])['runs']
    .transform(lambda x: x.expanding().mean().shift())
)

# 6 Handle Initial Values
# Use 0 for the first time a player plays at a venue. 
# Avoid using innings['savg'].mean() as it introduces "future" data leakage.
innings['savg'] = innings['savg'].fillna(0)

print(innings)

       match_id          batter  runs  out  cum_runs  cum_outs       pavg  \
0        548346  A Ashish Reddy    10    1       NaN       NaN        NaN   
1        548352  A Ashish Reddy     3    1      10.0       1.0  10.000000   
2        548359  A Ashish Reddy     8    1      13.0       2.0   6.500000   
3        548373  A Ashish Reddy    10    0      21.0       3.0   7.000000   
4        548376  A Ashish Reddy     4    1      31.0       3.0  10.333333   
...         ...             ...   ...  ...       ...       ...        ...   
16503    980903          Z Khan     4    1     107.0      12.0   8.916667   
16504    980993          Z Khan     2    1     111.0      13.0   8.538462   
16505   1082595          Z Khan     1    0     113.0      14.0   8.071429   
16506   1082635          Z Khan     2    1     114.0      14.0   8.142857   
16507   1082646          Z Khan     1    0     116.0      15.0   7.733333   

         lfm                               venue      savg  
0        NaN  

In [23]:
innings['pavg'] = innings['pavg'].fillna(0)
innings['lfm'] = innings['lfm'].fillna(0)
innings['savg'] = innings['savg'].fillna(0)


print(innings)

       match_id          batter  runs  out  cum_runs  cum_outs       pavg  \
0        548346  A Ashish Reddy    10    1       NaN       NaN   0.000000   
1        548352  A Ashish Reddy     3    1      10.0       1.0  10.000000   
2        548359  A Ashish Reddy     8    1      13.0       2.0   6.500000   
3        548373  A Ashish Reddy    10    0      21.0       3.0   7.000000   
4        548376  A Ashish Reddy     4    1      31.0       3.0  10.333333   
...         ...             ...   ...  ...       ...       ...        ...   
16503    980903          Z Khan     4    1     107.0      12.0   8.916667   
16504    980993          Z Khan     2    1     111.0      13.0   8.538462   
16505   1082595          Z Khan     1    0     113.0      14.0   8.071429   
16506   1082635          Z Khan     2    1     114.0      14.0   8.142857   
16507   1082646          Z Khan     1    0     116.0      15.0   7.733333   

         lfm                               venue      savg  
0       0.00  

In [24]:


innings.to_csv("csv/list.csv", index=False)

In [25]:
print(list(innings.batter.unique()))

['A Ashish Reddy', 'A Badoni', 'A Chandila', 'A Chopra', 'A Choudhary', 'A Dananjaya', 'A Flintoff', 'A Kamboj', 'A Kumble', 'A Manohar', 'A Mishra', 'A Mithun', 'A Mukund', 'A Nehra', 'A Nortje', 'A Raghuvanshi', 'A Singh', 'A Symonds', 'A Tomar', 'A Uniyal', 'A Zampa', 'AA Bilakhia', 'AA Chavan', 'AA Jhunjhunwala', 'AA Kulkarni', 'AA Noffke', 'AB Agarkar', 'AB Barath', 'AB Dinda', 'AB McDonald', 'AB de Villiers', 'AC Blizzard', 'AC Gilchrist', 'AC Thomas', 'AC Voges', 'AD Hales', 'AD Mascarenhas', 'AD Mathews', 'AD Nath', 'AD Russell', 'AF Milne', 'AG Murtaza', 'AG Paunikar', 'AJ Finch', 'AJ Hosein', 'AJ Turner', 'AJ Tye', 'AK Markram', 'AL Menaria', 'AM Nayar', 'AM Rahane', 'AN Ahmed', 'AN Ghosh', 'AP Dole', 'AP Majumdar', 'AP Tare', 'AR Bawne', 'AR Patel', 'AS Joseph', 'AS Rajpoot', 'AS Raut', 'AS Roy', 'AS Yadav', 'AT Carey', 'AT Rayudu', 'AU Rashid', 'AUK Pathan', 'Abdul Basith', 'Abdul Samad', 'Abdur Razzak', 'Abhishek Sharma', 'Abishek Porel', 'Akash Deep', 'Akash Madhwal', 'Am

In [26]:


# 2. Define the renaming map for consistency across seasons
rename_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiant": "Rising Pune Supergiants"
}

# 3. Apply renaming to both dataframes
# Apply to matches.csv team columns
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(rename_map)

# Apply to df.csv team columns
for col in ['batting_team', 'bowling_team']:
    df[col] = df[col].replace(rename_map)

# 4. Merge df with matches to associate each delivery with its season
# matches['id'] corresponds to df['match_id']
merged_df = pd.merge(
    df, 
    matches[['id', 'season']], 
    left_on='match_id', 
    right_on='id'
)

# 5. Extract unique players (Batters and Bowlers) per team and season
# Get unique batters for each team in each season
batters = merged_df[['season', 'batting_team', 'batter']].rename(
    columns={'batting_team': 'team', 'batter': 'player'}
)

# Get unique bowlers for each team in each season
bowlers = merged_df[['season', 'bowling_team', 'bowler']].rename(
    columns={'bowling_team': 'team', 'bowler': 'player'}
)

# Combine both lists and remove duplicates to get unique Season-Team-Player entries
final_players_df = pd.concat([batters, bowlers], ignore_index=True).drop_duplicates()

# Sort the data for better readability
final_players_df = final_players_df.sort_values(by=['season', 'team', 'player'])

# 6. Save the final processed data to a CSV file
output_filename = 'csv/processed_team_players.csv'
final_players_df.to_csv(output_filename, index=False)

print(f"Processing complete. Data saved to {output_filename}")
print(final_players_df.head())

Processing complete. Data saved to csv/processed_team_players.csv
        season                 team           player
11805  2007/08  Chennai Super Kings         A Mukund
6582   2007/08  Chennai Super Kings    CK Kapugedera
4604   2007/08  Chennai Super Kings        JA Morkel
302    2007/08  Chennai Super Kings         JDP Oram
5568   2007/08  Chennai Super Kings  Joginder Sharma
